# Regression checks: executable examples

This notebook reproduces problems that were reported during the peer review of the CLABTOOLKIT manuscript and shows that each one is fixed. The cells also work as regression checks: if one of them fails after a change in the toolkit, the problem has come back. Every example uses the sample data of clabtoolkit, found with `get_sample_data_dir` (the surfaces and cortical maps are downloaded from Zenodo the first time they are needed, and the annotations and the parcellation are shipped with the package), or simulated data. Only A7 needs FreeSurfer, because it trains a classifier with FreeSurfer's `mris_ca_train`: it prints a message and skips itself when `FREESURFER_HOME` is not set. Each code cell is self-contained and can be run on its own.

## Contents

**Part A. Checks of the reported issues**

| Cell | Topic | Problem that was fixed | FreeSurfer |
|---|---|---|---|
| [A1](#a1) | `BrainPlotter` themes and figure configuration | Plotting failed on a stock installation | no |
| [A2](#a2) | `Surface.load_scalar_maps` | Scalar maps from a DataFrame were not loaded | no |
| [A3](#a3) | `get_regionwise_timeseries` and `RegionTimeSeries` | `RegionTimeSeries` did not behave like an array | no |
| [A4](#a4) | `build_indices` with tuples | Tuples longer than two elements were rejected | no |
| [A5](#a5) | `get_subjects` on a BIDS dataset | Duplicate subjects and no check of the root folder | no |
| [A6](#a6) | `AnnotParcellation.group_into_lobes` | Method returned `None` | no |
| [A7](#a7) | `AnnotParcellation.annot2gcs` | Wrong `sphere.reg` file name | yes |
| [A8](#a8) | `AnnotParcellation.annot2gii` | Example used a removed `ref_surf` argument | no |


---
# Part A. Checks of the reported issues

Each cell below reproduces one of the problems reported during the review and shows that it is fixed.

<a id="a1"></a>

## A1. `BrainPlotter` themes and figure configuration

Broken visualization functionality. The cell plots three maps (annotation, curvature and thickness) of the sample surface, lists the layouts and themes, applies the `dark` theme and resets the figure configuration.

In [ ]:
# Answer (Reviewer 1. Question 2)

########################### Testing preview_theme, apply_theme, and figure configuration ###########################
import clabtoolkit.visualizationtools as cltvis
import clabtoolkit.surfacetools as cltsurf
import clabtoolkit.misctools as cltmisc
import os

# Sample data of clabtoolkit (the surfaces and the maps are downloaded from Zenodo the first time; the annotations are shipped with the package)
surf_dir = cltmisc.get_sample_data_dir("surfaces")
annot_dir = cltmisc.get_sample_data_dir("annotations")
maps_dir = cltmisc.get_sample_data_dir("corticalmaps")
prefix = "sub-colin27_ses-01_hemi-L_space-tallin"
surf_file = os.path.join(surf_dir, f"{prefix}_pial.surf.gii")
annot_file = os.path.join(annot_dir, f"{prefix}_atlas-Desikan2006_dseg.label.gii")
ct_map_file = os.path.join(maps_dir, f"{prefix}_thickness.shape.gii")
curv_map_file = os.path.join(maps_dir, f"{prefix}_curv.shape.gii")

# Load the surface geometry
surf = cltsurf.Surface(surf_file, color="#005AD8")

# Load the annotation
surf.load_annotation(annot_file, parc_name="Desikan")

# Load the vertex-wise maps
surf.load_scalar_maps(curv_map_file, maps_names="Curvature")
surf.load_scalar_maps(ct_map_file, maps_names="Thickness")

# Initialize the brain plotter
plotter = cltvis.BrainPlotter()
plotter.plot(surf, 
            map_names=["Desikan", "Curvature", "Thickness"],
            colorbar_titles=["Curvature (a.u.)", "Thickness (mm)"],
            views="lateral",
            use_opacity=False)  # Initial plot with the default figure configuration

print("Test 1: Printing layout details...")
plotter.get_layout_details('8_views')
print("-------------------------------")
print(" ")

print("Test 2: List all available themes...")
plotter.list_available_themes()
print("-------------------------------")
print(" ")

print("Test 3: Printing the characteristics of the 'dark' theme...")
plotter.preview_theme('dark')
print("-------------------------------")
print(" ")

print("Test 4: Applying the 'dark' theme...")
plotter.apply_theme('dark')
print("-------------------------------")
print(" ")

# Plot again with the applied theme
plotter.plot(surf,
            map_names=["Desikan", "Curvature", "Thickness"],
            colorbar_titles=["Curvature (a.u.)", "Thickness (mm)"],
            views="lateral",
            use_opacity=False)  # Plot with the applied 'dark' theme
print("------ Printing the figure configuration ------")
plotter.get_figure_config()
print("-------------------------------")
print(" ")

print("Test 5: Resetting the figure configuration...")
plotter.reset_figure_config()
plotter.plot(surf,
            map_names=["Desikan", "Curvature", "Thickness"],
            views="lateral",
            colorbar_titles=["Curvature (a.u.)", "Thickness (mm)"],
            use_opacity=False)  # Plot after resetting the figure configuration
print("------ Printing the figure configuration after reset ------")
plotter.get_figure_config()
print("-------------------------------")
print(" ")


<a id="a2"></a>

## A2. `Surface.load_scalar_maps`

Vertex-wise and region-wise maps are loaded from a DataFrame, a CSV file, NumPy arrays and map files. The loaded maps are listed after each test. The surface, the annotation and the maps come from the sample data of clabtoolkit.

In [ ]:
# Answer (Reviewer 1. Question 5)

########################### Testing load_scalar_maps ###########################
import tempfile
import clabtoolkit.surfacetools as cltsurf
import clabtoolkit.freesurfertools as cltfree
import clabtoolkit.misctools as cltmisc
import numpy as np
import pandas as pd
import os

# Sample data of clabtoolkit (the surfaces and the maps are downloaded from Zenodo the first time; the annotations are shipped with the package)
surf_dir = cltmisc.get_sample_data_dir("surfaces")
annot_dir = cltmisc.get_sample_data_dir("annotations")
maps_dir = cltmisc.get_sample_data_dir("corticalmaps")
prefix = "sub-colin27_ses-01_hemi-L_space-tallin"
surf_file = os.path.join(surf_dir, f"{prefix}_pial.surf.gii")
annot_file = os.path.join(annot_dir, f"{prefix}_atlas-Desikan2006_dseg.label.gii")
ct_map_file = os.path.join(maps_dir, f"{prefix}_thickness.shape.gii")
curv_map_file = os.path.join(maps_dir, f"{prefix}_curv.shape.gii")

# Loading the surface
surf_obj = cltsurf.Surface(surf_file)

# Loading the parcellation
parc = cltfree.AnnotParcellation(annot_file)

####################################################
print(f'Number of regions in the parcellation: {len(parc.regnames)}')

# Create a random dataframe with one row per region and the columns region_index, cthickness, curvature, area,
# volume, sulcal_depth, gyrification and pvalue (all the values are random).
# This is just an example, you can modify the values as needed.
# Each column is a 1D array with one value per region
np.random.seed(42)  # For reproducibility
cthickness = np.random.uniform(0.1, 5.0, size=len(parc.regnames))
curvature = np.random.uniform(-1.0, 1.0, size=len(parc.regnames))
area = np.random.uniform(1000, 5000, size=len(parc.regnames))
volume = np.random.uniform(5000, 20000, size=len(parc.regnames))
sulcal_depth = np.random.uniform(0.1, 2.0, size=len(parc.regnames))
gyrification = np.random.uniform(0.1, 2.0, size=len(parc.regnames))

# Create a DataFrame with these values
values_df = pd.DataFrame({
    'region_index': np.arange(len(parc.regnames)),
    'cthickness': cthickness,
    'curvature': curvature,
    'area': area,
    'volume': volume,
    'sulcal_depth': sulcal_depth,
    'gyrification': gyrification,
    'pvalue': np.random.uniform(0.0, 1.0, size=len(parc.regnames))
})

# save values to a csv file
values_df.to_csv(os.path.join(tempfile.gettempdir(), "values.csv"), index=False)
####################################################

# Test 1: Load a vertex-wise morphometry map
print("\nTest 1: Loading the map and using it as overlay")
surf_obj.load_scalar_maps(ct_map_file, maps_names="Thickness")  # Assign a name to the map. It will be used as overlay name
surf_obj.plot(maps="Thickness", hemi="lh", colorbar_title="Cortical Thickness (mm)")
print('')

# Test 2: Using just the dataframe without saving it to a CSV file.
# The dataframe has the same number of rows as the number of regions
print('Test 2: Using just the dataframe without saving it to a CSV file.')
surf_obj = cltsurf.Surface(surf_file)

# The names of the columns will be used as maps_names
surf_obj.load_scalar_maps(values_df, annotation=annot_file) # Using the dataframe directly
print(" ")

print("Overlays loaded from the dataframe:")
overlays = surf_obj.list_overlays()
print(overlays.keys())
print(" ")
print("Info about the 'volume' overlay:")

info = surf_obj.get_overlay_info("volume")
print(info)
print('------------------------------------------------------')
print(" ")


# Test 3: Using the dataframe saved to a CSV file
# The dataframe has the same number of rows as the number of regions
print('Test 3: Using the CSV file.')
surf_obj = cltsurf.Surface(surf_file)
surf_obj.load_scalar_maps(scalar_map=os.path.join(tempfile.gettempdir(), "values.csv"), annotation=annot_file) # Using the CSV file
print(" ")

print("Overlays loaded from the CSV file:")
overlays = surf_obj.list_overlays()
print(overlays.keys())
print(" ")

print("Info about the 'curvature' overlay:")
info = surf_obj.get_overlay_info("curvature")
print(info)
print('------------------------------------------------------')
print(" ")

surf_obj.plot(maps="pvalue", hemi="lh", cmap="hot")

# Test 4: Loading from a numpy array
print(" ")
print('Test 4: Loading from a numpy array')
surf_obj = cltsurf.Surface(surf_file)

# Converting dataframe to numpy array and loading the maps with annotation
surf_obj.load_scalar_maps(values_df.to_numpy(), 
                                annotation=annot_file, maps_names=list(values_df.columns))
print(" ")
print("Overlays loaded from the numpy array:")
overlays = surf_obj.list_overlays()
print(overlays.keys())
print(" ")
print("Info about the 'gyrification' overlay:")
info = surf_obj.get_overlay_info("gyrification")
print(info)
print('------------------------------------------------------')
print(" ")
surf_obj.plot(maps="gyrification", hemi="lh", 
                colorbar_title="Gyrification Index (a.u.)")


########################### Testing load_scalar_maps with vertex-wise dataframes or csv ###########################
######### Creating a csv with values as the number of vertices
n_points = surf_obj.mesh.n_points
values_df = pd.DataFrame({'vertex_index': np.arange(n_points), 'vertex_value': np.random.rand(n_points)})

values_df.to_csv(os.path.join(tempfile.gettempdir(), "values-vertexwise.csv"), index=False)

# Test 5: Loading vertex-wise maps from a CSV file
print("Test 5: Loading vertex-wise maps from a CSV file")
surf_obj.load_scalar_maps(os.path.join(tempfile.gettempdir(), "values-vertexwise.csv"))
print("Loaded vertex-wise maps from CSV file:")
overlays = surf_obj.list_overlays()
print(overlays.keys())
print(" ")
print('------------------------------------------------------')
print("")

# Test 6: Reading a vertex-wise map from a numpy array and a specified name
print("Test 6: Reading a vertex-wise map from a numpy array with a specified name")
values_array = np.random.rand(n_points)
surf_obj.load_scalar_maps(values_array,
                                maps_names=["ex5_vertex_value_array"])
print("Loaded vertex-wise map from numpy array:")
overlays = surf_obj.list_overlays()
print(overlays.keys())
print('------------------------------------------------------')
print("")

# Test 7: Creating a numpy array without specifying names
values_array = np.random.rand(n_points)
print("Test 7: Creating a numpy array with values as the number of vertices without specifying names")
surf_obj.load_scalar_maps(values_array)
print("Loaded vertex-wise maps from numpy array without specifying names:")
overlays = surf_obj.list_overlays()
print(overlays.keys())
print('------------------------------------------------------')
print("")

# Test 8: Reading multiple map files
print("Test 8: Reading multiple map files specifying names")
list_of_maps = [ct_map_file, curv_map_file]
maps_names = ["thickness2", "curvature2"]

for i, map_file in enumerate(list_of_maps):
    surf_obj.load_scalar_maps(map_file, maps_names=maps_names[i])
print("Loaded vertex-wise maps from map files:")
overlays = surf_obj.list_overlays()
print(overlays.keys())
print('------------------------------------------------------')
print("")


<a id="a3"></a>

## A3. `get_regionwise_timeseries` and `RegionTimeSeries`

Ten regions of the sample parcellation and a simulated 4D image on its grid give a `RegionTimeSeries` object. The cell shows the array in `.data`, the new `.shape`, the region names and colors, and the summary printed by `get_info()`.

In [ ]:
# Answer (Reviewer 1. Question 6)

########################### Testing get_regionwise_timeseries ###########################
import tempfile
import clabtoolkit.parcellationtools as cltparc
import clabtoolkit.imagetools as cltimg
import clabtoolkit.visualizationtools as cltvis
import clabtoolkit.misctools as cltmisc
import os


# Sample data of clabtoolkit (the parcellation is shipped with the package)
parc = cltparc.Parcellation(os.path.join(cltmisc.get_sample_data_dir("parcellations"), "sub-colin27_ses-01_space-tallin_atlas-chimeraDFMFHISIFN_seg-aparc_desc-grow1mm_dseg.nii.gz"))
# Ten regions: the precentral and postcentral cortex, the amygdala, the hippocampus and the cerebellum of both hemispheres
parc.keep_by_name(['ctx-rh-precentral', 'ctx-lh-precentral', 'ctx-rh-postcentral', 'ctx-lh-postcentral',
                   'amygd-rh-amygdala', 'amygd-lh-amygdala', 'hipp-rh-hippocampus-body', 'hipp-lh-hippocampus-body',
                   'cer-rh-i_iv', 'cer-lh-i_iv'])

# Simulate time-series data (there is no fMRI image in the sample data)
sim_image = os.path.join(tempfile.gettempdir(), 'simulated_4d_data.nii.gz')
sim_img = cltimg.simulate_image(
    reference=parc,
    simulated_image=sim_image,
    n_volumes=20
)

# Example 1: Compute region-wise time series from file and removing the first 3 volumes
# NOTE: get_regionwise_timeseries returns a RegionTimeSeries object (not a raw numpy
# array). The actual (n_regions x timepoints) array lives in its `.data` attribute,
# and the region metadata lives in `.region_names` / `.region_colors`.
print("")
print('Test 1: Computing region-wise time series from file...')
region_ts = parc.get_regionwise_timeseries(sim_image, method="clabtoolkit", vols_to_delete="1-3")
print("Region-wise time series shape:", region_ts.data.shape)
cltvis.create_carpet_plot(region_ts, figsize=(9, 6), tr=2.0, title="Region-wise Time Series Carpet Plot (clabtoolkit)")

print(" ")
print("-------------------------------")
print(f"This is the shape of the region-wise time series: {region_ts.shape}")
print(f"Region names: {region_ts.region_names}")
print(f"Region colors: {region_ts.region_colors}")

print(" ")
print("-------------------------------")
region_ts.get_info()


<a id="a4"></a>

## A4. `build_indices` with tuples

A mixed list of integers, tuples, lists, arrays and range strings, the tuple `(1, 2, 3, 4, 5)` from the report, and the nested input that is not supported and raises an error.

In [ ]:
# Answer (Reviewer 1. Question 7)

######################## Testing build_indices ###################################
import clabtoolkit.misctools as cltmisc
import numpy as np
# Test with a list of indices including integers, tuples, lists, and strings
print('Test 1: List of indices including integers, tuples, lists, and strings')
range_vector = [1, (2, 5), [6, 7], np.array([0, 0, 0]),  "8-10", "11:13", "14:2:22"]
indexes = cltmisc.build_indices(range_vector)
print(f'Indexes: {indexes}')
print('')

# Previous test was with a list of indices, now testing with a tuple representing a range of integers
print('Test 2: Test tuple with range of integers')
range_vector = (3,10)
indexes = cltmisc.build_indices(range_vector)
print(f'Indexes: {indexes}')
print('')

# Testing the reviewer's example with a simple tuple of integers
print("Test 2: The reviewer's example")
range_vector = (1,2,3,4,5)
indexes = cltmisc.build_indices(range_vector)
print(f'Indexes: {indexes}')
print('')

# Testing a fail example with nested inputs (list of lists)
print('Test 3: Fail example')
range_vector= ([1,2],[4,5])
try:
    indexes = cltmisc.build_indices(range_vector)
    print(f'Indexes: {indexes}')
except Exception as e:
    print(f'Error raised, as expected: nested inputs (list of lists, tuple of lists or tuple of tuples) are not supported ({e})')
print('')


<a id="a5"></a>

## A5. `get_subjects` on a BIDS dataset

A simulated BIDS dataset with derivatives is created, and the subjects are listed: only the top-level `sub-*` folders are returned, without duplicates.

In [ ]:
# Answer (Reviewer 1. Question 8)

######################## Testing get_subjects ###################################

import tempfile
import os
import clabtoolkit.bidstools as cltbids

##############################################################################
bids_dir = os.path.join(tempfile.gettempdir(), "sim_bids_test10")   # Folder of the simulated dataset (it is overwritten: do not point it to real data)

# Create a simulated BIDS dataset for testing purposes
cltbids.create_a_simulated_bids_dataset(
        output_dir=bids_dir,
        n_subjects=10,
        n_visits=2,
        modalities=["anat", "func"],
        tasks=["rest", "nback"],
        include_derivatives=["chimera", "fsl-first"],
        runs_per_modality=1,
        overwrite=True,
    )

# Testing get_subjects function to retrieve all subjects in the BIDS directory
print('Test 1: Get all subjects in the BIDS directory:')
subjects = cltbids.get_subjects(bids_dir)
print(f'Found {len(subjects)} subjects:')
print(subjects)
print('')

# Giving an empty folder
print('Test 2: Get subjects from an empty folder:')
empty_dir = os.path.join(tempfile.gettempdir(), "nonexistent_bids_test")
subjects = cltbids.get_subjects(empty_dir)
print(f'Found {len(subjects)} subjects:')
print(subjects)
print('')

<a id="a6"></a>

## A6. `AnnotParcellation.group_into_lobes`

Groups the Desikan regions of the sample annotation into lobes and returns the lobar annotation, which is saved and plotted.

In [ ]:
# Answer (Reviewer 1. Question 9a)

######################## Testing group_into_lobes ###################################
# The lobe definitions refer to Desikan-Killiany region names.
import tempfile
import os

import clabtoolkit.freesurfertools as cltfree
import clabtoolkit.surfacetools as cltsurf
import clabtoolkit.misctools as cltmisc

# Sample data of clabtoolkit (the surface is downloaded from Zenodo the first time; the annotation is shipped with the package)
hemi_id = 'lh'
annot_id = 'Desikan'
annot_file = os.path.join(cltmisc.get_sample_data_dir("annotations"), "sub-colin27_ses-01_hemi-L_space-tallin_atlas-Desikan2006_dseg.label.gii")
annot_parc = cltfree.AnnotParcellation(parc_file=annot_file, annot_id=annot_id)
surface = cltsurf.Surface(os.path.join(cltmisc.get_sample_data_dir("surfaces"), "sub-colin27_ses-01_hemi-L_space-tallin_pial.surf.gii"))

out_annot = os.path.join(tempfile.gettempdir(), f'{hemi_id}.desikan_lobes.annot')
print(f'Test 1: Grouping into lobes and saving to {out_annot}...')
lobar_parc = annot_parc.group_into_lobes(
    grouping='desikan',
    out_annot=out_annot,
    ctxprefix=f'{hemi_id}_',
    overwrite=True,
)
print(f'  File saved: {os.path.isfile(out_annot)}')
print(f'  Lobe names: {lobar_parc.regnames}')
print('')

# Load the lobar annotation onto the surface and plot it
surface.load_annotation(annotation=out_annot, parc_name="lobes")
surface.plot(maps="lobes")

<a id="a7"></a>

## A7. `AnnotParcellation.annot2gcs`

Trains a `.gcs` classifier from the Desikan annotation using the hemisphere-specific `sphere.reg`, converts it back with `gcs2annot` and plots the result. Requires FreeSurfer.

In [ ]:
# Answer (Reviewer 1. Question 9b)

######################## Testing annot2gcs ###################################
# Requires FreeSurfer: trains a classifier from the fsaverage Desikan annotation.
import tempfile
import os

import clabtoolkit.freesurfertools as cltfree
import clabtoolkit.surfacetools as cltsurf

freesurfer_home = os.environ.get('FREESURFER_HOME')
if freesurfer_home is None:
    print("FreeSurfer installation not detected (FREESURFER_HOME is not set). Skipping this example.")
else:
    subjects_dir = os.path.join(freesurfer_home, 'subjects')
    subject_id = 'fsaverage'
    hemi_id = 'lh'
    annot_id = 'aparc'
    annot_file = os.path.join(subjects_dir, subject_id, 'label', f'{hemi_id}.{annot_id}.annot')
    surf_file = os.path.join(subjects_dir, subject_id, 'surf', f'{hemi_id}.pial')
    surface = cltsurf.Surface(surf_file)

    out_gcs = os.path.join(tempfile.gettempdir(), f'{hemi_id}.{annot_id}_trained.gcs')

    # Test 1: Train a GCS classifier from the Desikan annotation
    print(f'Test 1: Training a GCS classifier from {os.path.basename(annot_file)} using {subject_id}...')
    annot_parc = cltfree.AnnotParcellation(parc_file=annot_file, annot_id='Desikan')
    gcs_name = annot_parc.annot2gcs(
        gcs_file=out_gcs,
        freesurfer_dir=subjects_dir,
        fssubj_id=subject_id,
        hemi=hemi_id,
    )
    print(f'  GCS filename: {gcs_name}')
    print(f'  File exists : {os.path.isfile(out_gcs)}')
    print('')

    # Converting it back to an annotation file to plot it on the surface
    out_annot = os.path.join(tempfile.gettempdir(), f'{hemi_id}.{annot_id}_from_gcs.annot')
    annot_parc.gcs2annot(
        gcs_file=out_gcs,
        annot_file=out_annot,
        freesurfer_dir=subjects_dir,
        ref_id=subject_id,
    )
    print(f'  Converted annotation file: {out_annot}')
    print(f'  File exists : {os.path.isfile(out_annot)}')

    # Load the converted annotation onto the surface and plot it. The black medial wall represents regions not covered by the annotation.
    surface.load_annotation(annotation=out_annot, parc_name="from_gcs")
    surface.plot(maps="from_gcs")


<a id="a8"></a>

## A8. `AnnotParcellation.annot2gii`

Converts an annotation to a GIFTI label file with the current signature (no reference surface), loads it onto the surface and plots it. The sample surface and annotation are used.

In [ ]:
# Answer (Reviewer 1. Question 9c)

######################## Testing annot2gii (static method) ###################################
import tempfile
import os
import clabtoolkit.freesurfertools as cltfree
import clabtoolkit.surfacetools as cltsurf
import clabtoolkit.misctools as cltmisc

############## Loading example Annotation ###################################
# Sample data of clabtoolkit (the surface is downloaded from Zenodo the first time; the annotation is shipped with the package)
sample_annot = os.path.join(cltmisc.get_sample_data_dir("annotations"), "sub-colin27_ses-01_hemi-L_space-tallin_atlas-Desikan2006_dseg.label.gii")
surface = cltsurf.Surface(os.path.join(cltmisc.get_sample_data_dir("surfaces"), "sub-colin27_ses-01_hemi-L_space-tallin_pial.surf.gii"))
annot_id = 'Desikan'
hemi_id = 'lh'
annot_parc = cltfree.AnnotParcellation(parc_file=sample_annot, annot_id=annot_id)

# The example converts a FreeSurfer .annot file, so the sample annotation is saved as one
annot_file = os.path.join(tempfile.gettempdir(), f'{hemi_id}.{annot_id}.Test.annot')
annot_parc.save_annotation(annot_file)

# Test 1: Convert with an explicit output file
gii_file = os.path.join(tempfile.gettempdir(), f'{hemi_id}.{annot_id}_test.label.gii')
print(f'Test 1: Converting {os.path.basename(annot_file)} to {gii_file}...')
cltfree.AnnotParcellation.annot2gii(annot_file=annot_file, gii_file=gii_file)
print(f'  File exists: {os.path.isfile(gii_file)}')
print('')

# Load the converted GIFTI label file onto the surface and plot it
surface.load_annotation(annotation=gii_file, parc_name="test")
surface.plot(maps="test")
